# Advanced EDA - CWRU Bearing Diagnostics

This notebook provides a professional-grade diagnostic analysis of the CWRU bearing dataset. 

### Key Modules:
1. **Multi-Fault Comparison**: Side-by-side analysis of all fault types.
2. **Theoretical Frequency Analysis**: Calculated based on SKF 6205/6203 geometry.
3. **Advanced Signal Processing**: Envelope Spectrum & Spectrograms.
4. **Wavelet Packet Energy**: Analyzing energy distribution across 8 frequency bands.
5. **Feature Intelligence**: Deep-dive into class separability across loads.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import scipy.io
from scipy.fft import fft, fftfreq
from scipy.signal import hilbert, spectrogram

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams['figure.figsize'] = (14, 7)

# Sample Paths
DATA_DIR = "../data/raw"
SAMPLES = {
    'Normal': os.path.join(DATA_DIR, "normal/Normal_0.mat"),
    'Inner Race (IR)': os.path.join(DATA_DIR, "12k_drive_end/IR007_0.mat"),
    'Ball (B)': os.path.join(DATA_DIR, "12k_drive_end/B007_0.mat"),
    'Outer Race (OR)': os.path.join(DATA_DIR, "12k_drive_end/OR007@6_0.mat")
}

PROCESSED_PATH = "../data/processed/cwru_features.parquet"
SPECS_PATH = "../data/external/bearing_specs.csv"

## 1. Multi-Fault Comparison (Raw Waveforms)
Comparing the characteristic "signatures" of different faults in the time domain.

In [ ]:
def load_raw_vibration(path, sensor='DE'):
    mat = scipy.io.loadmat(path)
    key = [k for k in mat.keys() if f'_{sensor}_time' in k][0]
    return mat[key].flatten()

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

for i, (label, path) in enumerate(SAMPLES.items()):
    data = load_raw_vibration(path)
    axes[i].plot(data[:2000], linewidth=0.8)
    axes[i].set_title(f"{label} Fault Vibration")
    axes[i].set_ylabel("Amplitude (g)")

axes[3].set_xlabel("Sample Index")
plt.tight_layout()
plt.show()

## 2. Advanced: Characteristic Fault Frequencies
Frequencies (BPFO, BPFI, BSF) based on bearing geometry (diameter, balls) and shaft speed (RPM).

In [ ]:
def get_fault_frequencies(rpm, pd_val, bd_val, n_balls, alpha_deg=0):
    fr = rpm / 60.0
    alpha = np.radians(alpha_deg)
    bpfo = (n_balls / 2.0) * fr * (1 - (bd_val / pd_val) * np.cos(alpha))
    bpfi = (n_balls / 2.0) * fr * (1 + (bd_val / pd_val) * np.cos(alpha))
    bsf = (pd_val / (2.0 * bd_val)) * fr * (1 - (bd_val / pd_val)**2 * np.cos(alpha)**2)
    return {"BPFO": bpfo, "BPFI": bpfi, "BSF": bsf}

freqs_6205 = get_fault_frequencies(rpm=1797, pd_val=39.04, bd_val=7.94, n_balls=9)
print("Theoretical Fault Frequencies (SKF 6205 @ 1797 RPM):")
for k, v in freqs_6205.items(): print(f" - {k}: {v:.2f} Hz")

## 3. Envelope Spectrum Analysis
Revealing periodic impacts after removing the high-frequency carrier.

In [ ]:
def plot_envelope_spectrum(sig, fs=12000, title="Envelope Spectrum", fault_freqs=None):
    analytic_signal = hilbert(sig)
    envelope = np.abs(analytic_signal)
    envelope -= np.mean(envelope)
    n = len(envelope)
    yf = fft(envelope)
    xf = fftfreq(n, 1/fs)[:n//2]
    mag = 2.0/n * np.abs(yf[0:n//2])

    plt.plot(xf, mag)
    plt.xlim(0, 500)
    plt.title(title)
    plt.xlabel("Frequency (Hz)")
    if fault_freqs:
        for name, f in fault_freqs.items():
            plt.axvline(x=f, color='r', linestyle='--', alpha=0.6, label=f"{name} ({f:.1f}Hz)")
        plt.legend()

ir_sig = load_raw_vibration(SAMPLES['Inner Race (IR)'])
plt.figure(figsize=(14, 6))
plot_envelope_spectrum(ir_sig, fault_freqs={'BPFI': freqs_6205['BPFI']}, title="Envelope Spectrum - Inner Race Fault")
plt.show()

## 4. Wavelet Packet Energy Distribution
Analyzing energy density across 8 frequency bands (Level 3 WPD). This shows where the fault energy is most concentrated.

In [ ]:
df = pd.read_parquet(PROCESSED_PATH)
wpd_cols = [c for c in df.columns if 'wpd_energy_node' in c]

wpd_melted = df.melt(id_vars=['fault_type'], value_vars=wpd_cols, var_name='WPD_Node', value_name='Relative_Energy')

plt.figure(figsize=(14, 7))
sns.barplot(data=wpd_melted, x='WPD_Node', y='Relative_Energy', hue='fault_type')
plt.title("Wavelet Packet Energy Distribution across 8 Bands")
plt.xticks(rotation=45)
plt.show()

## 5. Feature Intelligence: Load Condition Impact
How well features separate classes under varying load (0-3 HP).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.boxplot(data=df, x='load', y='rms', hue='fault_type', ax=axes[0])
axes[0].set_title("RMS vs Load Condition")
sns.boxplot(data=df, x='load', y='kurtosis', hue='fault_type', ax=axes[1])
axes[1].set_title("Kurtosis vs Load Condition")
plt.show()